# M3 – Encoding & Feature Selection (Regression)

**Project:** IT3051 FDM Mini Project 2026 – DataCo Smart Supply Chain  
**Owner:** M3 – Primesh (Encoding & Frontend)  
**Task:** Regression — predict `Days for shipping (real)`  
**Module:** `src/encoders.py`

| Step | What happens |
|------|--------------|
| 0 | Setup — imports, paths, the encoding module |
| 1 | Load the training data |
| 2 | Analyze categorical columns |
| 3 | Check Target Encoding for leakage |
| 4 | Compare encoding strategies for high-cardinality features |
| 5 | Feature selection — remove redundant, measure importance, backward selection |
| 6 | Save the final decisions |
| 7 | Perform a final test-set check |
| 8 | Viva summary |

**Before running:** `data/processed/train_reg.csv` and `test_reg.csv` must exist (produced by M1's notebook).

---
## Step 0 – Setup

**Set up the environment**

Import the required libraries, define file paths, and set the constants used throughout the notebook. Set `FAST = True` while developing to run on 30% of orders; set it back to `False` for the final run.

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
PROCESSED_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
FIG_DIR = ROOT / "reports" / "figures" / "m3"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "Days for shipping (real)"
GROUP = "Order Id"
SEED = 42
FAST = False          # True = use 30% of orders for a quick run while developing

def show(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=120)
    plt.show()

**The encoding module — `src/encoders.py`**

All encoding logic lives in `src/encoders.py`, not in this notebook, so the trained pipeline can be saved and loaded by the backend. The key function used throughout is `build_preprocessor(X, features, high_card)`, which builds a `ColumnTransformer` that applies the right encoding to each column type automatically.

| Function / Class | Purpose |
|------------------|---------|
| `LOW_CARD_MAX = 30` | Threshold — columns with ≤ 30 unique values are one-hot encoded; columns with more are treated as high-cardinality |
| `ENGINEERED` | Names the three M2 feature groups (`date_features`, `order_features`, `geo_features`) and the raw columns each one needs |
| `FrequencyEncoder` | Replaces each category with its share of training rows; unseen categories become 0 |
| `high_card_encoder(strategy)` | Returns the encoder object for a chosen high-cardinality strategy |
| `split_columns(X, features)` | Sorts a feature list into numeric, low-cardinality, and high-cardinality buckets |
| `build_preprocessor(X, features, high_card)` | Assembles the full `ColumnTransformer` — used in every experiment and in the final model |

**Four strategies for high-cardinality columns:**

| Strategy | What it does | Benefit | Risk |
|----------|-------------|---------|------|
| `drop` | Ignore the column | Simplest, zero risk | Loses any signal |
| `onehot_rare` | One-hot for categories covering ≥ 1% of rows; all rare ones share an `infrequent` column | Keeps common categories exactly | Still creates many columns |
| `frequency` | Replace each category with how often it appears in training | One number per column, no target used — no leakage risk | Two categories with the same frequency look identical |
| `target` | Replace each category with the average target value for that category | Directly captures each category's effect on delivery days | Can leak if fitted outside the pipeline — demonstrated in Step 3 |

---
## Step 1 – Load the training data

**Load the training data**

Load the `train_reg.csv` file prepared by M1, then separate the features `X`, target `y`, and `Order Id` groups. Create the 5 cross-validation folds once and reuse them in every experiment so all comparisons are made on identical data splits. Build `ALL_FEATURES` — every raw column plus M2's three engineered groups — as the starting candidate list.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from src.features import DATE_COL
from src.encoders import ENGINEERED, LOW_CARD_MAX, FrequencyEncoder, build_preprocessor, split_columns

train = pd.read_csv(PROCESSED_DIR / "train_reg.csv")
if FAST:
    keep = pd.Series(train[GROUP].unique()).sample(frac=0.3, random_state=SEED)
    train = train[train[GROUP].isin(keep)].reset_index(drop=True)

X, y, groups = train.drop(columns=[TARGET]), train[TARGET], train[GROUP]

# Same kind of folds as M1: grouped by order, stratified on the target. Created once, reused everywhere.
cv_splits = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y, groups))

ALL_FEATURES = [c for c in X.columns if c not in (GROUP, DATE_COL)] + [u for u in ENGINEERED if all(c in X.columns for c in ENGINEERED[u])]
print(f"Train: {len(X):,} rows, {groups.nunique():,} orders | {len(cv_splits)} grouped folds")
print("Candidate features:", ALL_FEATURES)

---
## Step 2 – Analyze categorical columns

### 2.1 Cardinality table

**Analyze categorical columns**

Check the cardinality of each categorical column. Columns with fewer than 30 unique values are treated as low-cardinality and will be one-hot encoded. Columns with more unique values are treated as high-cardinality and need a different strategy. The table also shows what percentage of categories are rare (fewer than 30 rows), which matters for reliability.

In [ ]:
NUMERIC, LOW_CARD, HIGH_CARD = split_columns(X, ALL_FEATURES)
CATEGORICAL = LOW_CARD + HIGH_CARD

def rare_share(s, min_rows=30):
    counts = s.value_counts()
    return (counts < min_rows).mean() * 100

card = pd.DataFrame({
    "n_categories": [X[c].nunique() for c in CATEGORICAL],
    "top_category_%": [X[c].value_counts(normalize=True).iloc[0] * 100 for c in CATEGORICAL],
    "rare_categories_%": [rare_share(X[c]) for c in CATEGORICAL],
    "group": ["low (one-hot)" if c in LOW_CARD else "HIGH" for c in CATEGORICAL],
}, index=CATEGORICAL).sort_values("n_categories", ascending=False)
card.round(1)

### 2.2 Encode low-cardinality features

**Encode low-cardinality features**

Use One-Hot Encoding for columns such as `Shipping Mode` and `Customer Segment` because they have only a small number of unique categories (≤ 30). One-Hot Encoding creates one binary column per category, which is manageable at this scale. High-cardinality columns like `Order City` have thousands of unique values and need a different approach — compared in Step 4.

| Encoding | Columns assigned |
|----------|------------------|
| **One-Hot** (`OneHotEncoder`) | `Shipping Mode` (4), `Type` (4), `Customer Segment` (3), `Market` (5), `Order Region` (22), `Department Name` (22), `Customer Country` (~3) |
| **High-cardinality strategy** (chosen in Step 4) | `Order City` (~3,500), `Order State` (~1,000), `Customer City` (~500), `Order Country` (~160), `Product Name` (~118), `Category Name` (~50), `Customer State` (~50) |
| **Numeric pass-through** (`SimpleImputer`) | `Benefit per order`, `Sales`, `Product Price`, `Latitude`, `Longitude`, `Order Item Quantity`, etc. |
| **M2 engineered groups** (numeric outputs) | `date_features` → 5 cols · `order_features` → 3 cols · `geo_features` → 1 col |

### 2.3 Long tail of the biggest column

**Long tail of the biggest column**

Plot the coverage curve for the column with the most categories. A slowly rising curve means most rows are spread across many rarely-occurring values — a long tail. This confirms that one-hot encoding is not suitable for high-cardinality columns, because most of the resulting columns would be almost always zero.

In [ ]:
col = card.index[0]                                    # the column with the most categories
coverage = X[col].value_counts(normalize=True).cumsum().reset_index(drop=True) * 100
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.arange(1, len(coverage) + 1), coverage)
ax.set_xlabel(f"Number of most common {col} values"); ax.set_ylabel("% of rows covered")
ax.set_title(f"Long tail of {col}")
show(fig, "01_long_tail")
for n in [10, 50, 100, 500]:
    if n <= len(coverage):
        print(f"Top {n:>4} values cover {coverage.iloc[n - 1]:.1f}% of rows")

### 2.4 Unseen categories in validation

**Unseen categories in validation**

Estimate how often a validation fold contains a category the training fold never saw. A non-zero percentage means the encoder must handle unknown values safely. This is also what happens in production — new orders arrive for cities or products the model has never seen. Every strategy in `src/encoders.py` has a built-in fallback for this case.

In [ ]:
# How often does a validation fold contain a category the training folds never saw?
unseen = {}
for c in HIGH_CARD:
    shares = []
    for tr_idx, va_idx in cv_splits:
        seen = set(X[c].iloc[tr_idx])
        shares.append((~X[c].iloc[va_idx].isin(seen)).mean() * 100)
    unseen[c] = np.mean(shares)
pd.Series(unseen, name="unseen_in_validation_%").round(2)

---
## Step 3 – Check Target Encoding for leakage

**Check Target Encoding for leakage**

Compare naive Target Encoding with safe Target Encoding inside a Pipeline. In the naive version, each city's average delivery days is calculated using all training rows — including the rows being validated — so the model effectively sees a version of the answer. In the correct version, `TargetEncoder` is placed inside a Pipeline so it is only fitted on the training fold during each cross-validation split. The difference in R² between the two versions is the leakage.

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder

SCORING = {"MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error", "R2": "r2"}
leak_col = card.index[0]

# WRONG: category means computed once on ALL training rows, then cross-validated
naive_feature = y.groupby(X[leak_col]).transform("mean").to_frame()
naive = cross_validate(LinearRegression(), naive_feature, y, cv=cv_splits, scoring=SCORING)

# RIGHT: TargetEncoder inside the pipeline -> learned from the training folds only
proper_pipe = Pipeline([("encode", TargetEncoder(target_type="continuous", random_state=SEED)),
                        ("model", LinearRegression())])
proper = cross_validate(proper_pipe, X[[leak_col]], y, cv=cv_splits, scoring=SCORING)

pd.DataFrame({
    "naive (leaky)": {"MAE": -naive["test_MAE"].mean(), "R2": naive["test_R2"].mean()},
    "inside pipeline": {"MAE": -proper["test_MAE"].mean(), "R2": proper["test_R2"].mean()},
}).round(4)

---
## Step 4 – Compare encoding strategies for high-cardinality features

### 4.1 Cross-validated comparison

**Compare encoding strategies for high-cardinality features**

Test all four encoding strategies — `drop`, `onehot_rare`, `frequency`, and `target` — using the same 5-fold `StratifiedGroupKFold` cross-validation and compare MAE, RMSE, and R². Two model types are used (Ridge and HistGradientBoosting) to check whether the best strategy holds for both a linear model and a tree model. The strategy with the lowest MAE is selected as `BEST_STRATEGY`.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor

STRATEGIES = ["drop", "onehot_rare", "frequency", "target"]
MODELS = {
    "Ridge": lambda: Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "HistGB": lambda: HistGradientBoostingRegressor(max_iter=200, random_state=SEED),
}

def evaluate(features, strategy, model_name):
    pipe = Pipeline([("prep", build_preprocessor(X, features, high_card=strategy)),
                     ("model", MODELS[model_name]())])
    s = cross_validate(pipe, X, y, cv=cv_splits, scoring=SCORING)
    return {"MAE": -s["test_MAE"].mean(), "RMSE": -s["test_RMSE"].mean(),
            "R2": s["test_R2"].mean(), "MAE_std": s["test_MAE"].std()}

rows = []
for strategy in STRATEGIES:
    for model_name in MODELS:
        rows.append({"strategy": strategy, "model": model_name, **evaluate(ALL_FEATURES, strategy, model_name)})
enc_results = pd.DataFrame(rows)
enc_results.round(4)

### 4.2 Comparison chart

**Comparison chart**

Plot the cross-validated MAE for each strategy and model side by side. The strategy with the shortest bars is the best choice. If bars are all similar in height, the high-cardinality columns carry little signal and the choice matters less.

In [ ]:
pivot = enc_results.pivot(index="strategy", columns="model", values="MAE").loc[STRATEGIES]
ax = pivot.plot.bar(figsize=(6, 3))
ax.set_ylabel("Cross-validated MAE (days)"); ax.set_title("Encoding strategy for high-cardinality columns")
ax.set_ylim(pivot.min().min() * 0.95, pivot.max().max() * 1.02)
show(ax.figure, "02_encoding_comparison")

### 4.3 Select the best encoding strategy

**Select the best encoding strategy**

Sort all results by MAE and store the winning strategy as `BEST_STRATEGY`. This value is used in every subsequent step — feature selection, the final preprocessor, and the saved `m3_selection.json`.

In [ ]:
best = enc_results.sort_values("MAE").iloc[0]
BEST_STRATEGY = best["strategy"]
print(f"Best: {BEST_STRATEGY} with {best['model']} (MAE {best['MAE']:.4f} ± {best['MAE_std']:.4f})")

---
## Step 5 – Feature selection

### 5.1 Remove redundant features

**Remove redundant features**

Find highly correlated numeric feature pairs with absolute Pearson correlation above 0.9 and remove the weaker duplicate — the one with the lower Spearman correlation to the target. Also check for near-constant categorical features where one category covers more than 99% of rows; these provide almost no useful information to the model. The remaining features become `CANDIDATES`.

In [ ]:
# Correlation filter: from each numeric pair with |r| > 0.9, drop the one less related to the target
num_corr = X[NUMERIC].corr()
target_rel = X[NUMERIC].corrwith(y, method="spearman").abs()
corr_drop = {}
for i, a in enumerate(NUMERIC):
    for b in NUMERIC[i + 1:]:
        r = num_corr.loc[a, b]
        if abs(r) > 0.9 and a not in corr_drop and b not in corr_drop:
            weaker = a if target_rel[a] < target_rel[b] else b
            stronger = b if weaker == a else a
            corr_drop[weaker] = f"|r| = {abs(r):.2f} with '{stronger}'"
print("Dropped by correlation filter:", corr_drop or "none")

In [ ]:
# Near-constant categorical columns: one category covers more than 99% of rows
near_constant = card.index[card["top_category_%"] > 99].tolist()
print("Near-constant columns:", near_constant or "none")

CANDIDATES = [f for f in ALL_FEATURES if f not in corr_drop and f not in near_constant]
print(f"{len(CANDIDATES)} candidate features after filters")

### 5.2 Measure feature importance

**Measure feature importance**

Use drop-column importance by removing one feature at a time and measuring how much the MAE changes on a held-out grouped fold. If removing a feature makes the MAE much worse, that feature is important. If removing it makes little or no difference — or even improves the MAE — the feature adds noise and can be dropped. M2's engineered groups (`date_features`, `order_features`, `geo_features`) are removed as whole units.

In [ ]:
from sklearn.metrics import mean_absolute_error

tr_idx, va_idx = cv_splits[0]

def fold_mae(features):
    pipe = Pipeline([("prep", build_preprocessor(X, features, high_card=BEST_STRATEGY)),
                     ("model", HistGradientBoostingRegressor(max_iter=200, random_state=SEED))])
    pipe.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    return mean_absolute_error(y.iloc[va_idx], pipe.predict(X.iloc[va_idx]))

base_mae = fold_mae(CANDIDATES)
importance = pd.Series(
    {f: fold_mae([g for g in CANDIDATES if g != f]) - base_mae for f in CANDIDATES},
    name="MAE_increase_when_removed",
).sort_values(ascending=False)
print(f"Validation MAE with all {len(CANDIDATES)} candidates: {base_mae:.4f}")
importance.round(4).to_frame()

### 5.3 Importance chart

**Importance chart**

Show the drop-column importance ranking as a horizontal bar chart. A bar extending to the right means the model gets worse without that feature. A bar at zero or to the left means the feature is unimportant or harmful. The zero line makes it easy to see which features are worth keeping.

In [ ]:
fig, ax = plt.subplots(figsize=(6, max(3, 0.3 * len(importance))))
ax.barh(importance.index[::-1], importance.values[::-1])
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("Increase in MAE when the feature is removed (days)"); ax.set_title("Drop-column importance")
show(fig, "03_drop_column_importance")

### 5.4 Perform backward feature selection

**Perform backward feature selection**

Test smaller feature sets — all features, top 12, top 8, top 5, top 3, and top 1 — ranked by drop-column importance from Step 5.2. Each set is scored with full 5-fold grouped cross-validation so the result is not biased by the single fold used for ranking. This shows whether removing weak features changes the MAE meaningfully.

In [ ]:
ranked = importance.index.tolist()
ks = sorted({len(ranked), 12, 8, 5, 3, 1} - {0}, reverse=True)
ks = [k for k in ks if k <= len(ranked)]

sel_rows = []
for k in ks:
    sel_rows.append({"k": k, "features": ranked[:k], **evaluate(ranked[:k], BEST_STRATEGY, "HistGB")})
selection = pd.DataFrame(sel_rows)
selection[["k", "MAE", "MAE_std", "R2"]].round(4)

### 5.5 Backward selection chart

**Backward selection chart**

Plot the cross-validated MAE with error bars against the number of features kept, from most on the left to fewest on the right. A flat line means the removed features were not contributing. A sharp jump shows the point where an important feature would be lost.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.errorbar(selection["k"], selection["MAE"], yerr=selection["MAE_std"], marker="o", capsize=3)
ax.invert_xaxis()
ax.set_xlabel("Number of features kept (most important first)"); ax.set_ylabel("CV MAE (days)")
ax.set_title("Does dropping weak features hurt?")
show(fig, "04_backward_selection")

### 5.6 Choose the final feature set

**Choose the final feature set**

Use the one-standard-error rule to select the smallest feature set that performs almost as well as the best one. Among all feature sets whose MAE is within one standard deviation of the best MAE, pick the one with the fewest features. This avoids overfitting, keeps the model easier to explain, and reduces the number of input fields required by the frontend.

In [ ]:
# Smallest feature set whose MAE is within one standard deviation of the best (the "one-standard-error" rule)
best_row = selection.loc[selection["MAE"].idxmin()]
ok = selection[selection["MAE"] <= best_row["MAE"] + best_row["MAE_std"]]
chosen = ok.loc[ok["k"].idxmin()]
SELECTED = chosen["features"]
dropped = {f: "low drop-column importance" for f in CANDIDATES if f not in SELECTED}
dropped.update({f: r for f, r in corr_drop.items()})
dropped.update({f: "near-constant" for f in near_constant})
print(f"Selected {len(SELECTED)} features (MAE {chosen['MAE']:.4f} vs best {best_row['MAE']:.4f}):")
print(SELECTED)

---
## Step 6 – Save the final decisions

**Save the final decisions**

Save the chosen encoding strategy, the selected feature list, the dropped features with their reasons, and the cross-validation MAE to `results/m3_selection.json`. This file is then read by `train_xgb.py`, M4's models, and the frontend to ensure every part of the system trains on exactly the same inputs. Also append all encoding and selection experiments to the shared `results/experiments.csv`.

In [ ]:
decision = {
    "target": TARGET,
    "high_card_strategy": BEST_STRATEGY,
    "low_card_max": LOW_CARD_MAX,
    "selected_features": list(SELECTED),
    "dropped_features": dropped,
    "cv": "StratifiedGroupKFold(5) grouped by Order Id",
    "selected_cv_MAE": round(float(chosen["MAE"]), 4),
}
(RESULTS_DIR / "m3_selection.json").write_text(json.dumps(decision, indent=2, ensure_ascii=False), encoding="utf-8")

log = pd.concat([
    enc_results.assign(owner="M3", experiment="encoding_" + enc_results["strategy"], n_features=len(ALL_FEATURES)),
    selection.assign(owner="M3", model="HistGB", experiment="selection_top" + selection["k"].astype(str), n_features=selection["k"]),
])[["owner", "experiment", "model", "n_features", "MAE", "MAE_std", "RMSE", "R2"]]
exp_path = RESULTS_DIR / "experiments.csv"
log.to_csv(exp_path, mode="a", header=not exp_path.exists(), index=False)
print("Saved m3_selection.json and", len(log), "rows to experiments.csv")

---
## Step 7 – Perform a final test-set check

**Perform a final test-set check**

Use the test set only at the end to confirm that the preprocessor can handle unseen data. Fit the final preprocessor on the full training data, then transform the test set. Check that train and test produce the same number of columns and that there are no missing values in the output. This step does not produce a test score — that is done once in PE2 for all models together.

In [ ]:
test = pd.read_csv(PROCESSED_DIR / "test_reg.csv")
final_prep = build_preprocessor(X, SELECTED, high_card=BEST_STRATEGY)
Xt_tr = final_prep.fit_transform(X, y)            # target encoder needs y when fitting
Xt_te = final_prep.transform(test.drop(columns=[TARGET]))
print("Train matrix:", Xt_tr.shape, "| Test matrix:", Xt_te.shape)
print("Missing values:", bool(np.isnan(Xt_tr).any() or np.isnan(Xt_te).any()))
print("Output columns:", list(final_prep.get_feature_names_out())[:15], "...")

---
## Step 8 – Viva summary

**Key results in one place**

Print a single summary block with all the numbers that may be asked about in the viva — which columns are high-cardinality, the leakage demo gap, the winning encoding strategy, what was dropped and why, the top features, and the final selected set with its MAE.

In [ ]:
print(f'''
HIGH-CARDINALITY  {", ".join(f"{c} ({X[c].nunique():,})" for c in HIGH_CARD)}
LOW-CARDINALITY   {", ".join(LOW_CARD)}  -> one-hot
LEAKAGE DEMO      naive target encoding R2 {naive["test_R2"].mean():.3f} vs inside pipeline {proper["test_R2"].mean():.3f}
ENCODING          best = {BEST_STRATEGY}  (CV MAE {best["MAE"]:.4f}, model {best["model"]})
CORR FILTER       dropped {list(corr_drop) or "none"}
NEAR-CONSTANT     dropped {near_constant or "none"}
TOP 3 FEATURES    {", ".join(ranked[:3])}
SELECTED          {len(SELECTED)} of {len(ALL_FEATURES)} features, CV MAE {chosen["MAE"]:.4f}
''')